# Real-World Experiment Training Curves

This notebook visualizes the training progress of real-world pose estimation experiments.


In [ ]:
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Style settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['legend.fontsize'] = 12

# Paths
CHECKPOINT_ROOT = Path("../../logs/pose_estimation/real_world/checkpoints")


In [ ]:
def load_epoch_history(checkpoint_dir: Path) -> dict:
    """Load epoch_history.json from a checkpoint directory."""
    history_path = checkpoint_dir / "epoch_history.json"
    if not history_path.exists():
        return None
    with open(history_path) as f:
        return json.load(f)

def find_experiment_dirs(root: Path, pattern: str = None) -> list:
    """Find all experiment directories matching a pattern."""
    if not root.exists():
        print(f"Warning: {root} does not exist")
        return []
    
    dirs = []
    for d in sorted(root.iterdir()):
        if d.is_dir():
            if pattern is None or pattern in d.name:
                if (d / "epoch_history.json").exists():
                    dirs.append(d)
    return dirs

def extract_experiment_name(dirname: str) -> str:
    """Extract readable experiment name from directory name."""
    # Format: PointTransformer-real_world_{name}_job00-{timestamp}
    parts = dirname.split("-")
    if len(parts) >= 2:
        name_part = parts[1]  # real_world_{name}_job00
        # Extract the middle part
        if name_part.startswith("real_world_"):
            name = name_part.replace("real_world_", "").replace("_job00", "")
            return name.capitalize()
    return dirname

# Find all experiment directories
exp_dirs = find_experiment_dirs(CHECKPOINT_ROOT)
print(f"Found {len(exp_dirs)} experiments:")
for d in exp_dirs:
    name = extract_experiment_name(d.name)
    history = load_epoch_history(d)
    n_epochs = len(history) if history else 0
    print(f"  - {name}: {n_epochs} epochs")


In [ ]:
def plot_training_curves(exp_dirs: list, metric: str = "mpjpe", split: str = "test"):
    """
    Plot training curves for multiple experiments.
    
    Args:
        exp_dirs: List of experiment directories
        metric: Metric to plot ('mpjpe', 'p_mpjpe', 'loss')
        split: Data split ('train' or 'test')
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(exp_dirs)))
    
    for i, exp_dir in enumerate(exp_dirs):
        history = load_epoch_history(exp_dir)
        if not history:
            continue
        
        name = extract_experiment_name(exp_dir.name)
        epochs = [h["epoch"] for h in history]
        values = [h[split][metric] for h in history]
        
        # Find best value
        best_idx = np.argmin(values) if metric != "loss" else np.argmin(values)
        best_epoch = epochs[best_idx]
        best_value = values[best_idx]
        
        # Plot line
        ax.plot(epochs, values, 'o-', color=colors[i], label=f"{name} (best: {best_value:.1f}mm @ epoch {best_epoch})", 
                linewidth=2, markersize=4)
        
        # Mark best point
        ax.scatter([best_epoch], [best_value], color=colors[i], s=150, zorder=5, 
                   edgecolors='black', linewidths=1.5)
    
    ax.set_xlabel("Epoch")
    ax.set_ylabel(f"{metric.upper()} (mm)" if metric != "loss" else "Loss")
    ax.set_title(f"{split.capitalize()} {metric.upper()} vs Epoch")
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig, ax


## Test MPJPE vs Epoch

Compare validation performance across different training configurations.


In [ ]:
# Plot Test MPJPE
plot_training_curves(exp_dirs, metric="mpjpe", split="test")
plt.savefig("test_mpjpe_curves.pdf", bbox_inches='tight', dpi=150)
plt.show()


## Train MPJPE vs Epoch

Training set performance (should decrease steadily).


In [ ]:
# Plot Train MPJPE
plot_training_curves(exp_dirs, metric="mpjpe", split="train")
plt.savefig("train_mpjpe_curves.pdf", bbox_inches='tight', dpi=150)
plt.show()
